# Ropedia Academy — A8 · Human–scene & human–object interaction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A8.ipynb)

> **Optimizes a body against contact + non-penetration constraints (physics as free supervision) and queries affordances — the human↔scene coupling end-to-end.**
>
> 在接触 + 非穿插约束下优化身体（物理即免费监督）并查询可供性——完整呈现人↔场景的耦合。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A8

In [ ]:
import torch

# A recovered body must be consistent with the scene: SUPPORTED + NO penetration.
body = torch.randn(50, 3, requires_grad=True)   # body vertices
foot = torch.tensor([0, 1])                      # two foot-vertex indices
def scene_sdf(p):                                # one spherical obstacle (signed dist)
    return (p - torch.tensor([0., 0.5, 0.])).norm(dim=-1) - 0.4   # <0 = inside

opt = torch.optim.Adam([body], lr=0.05)
for _ in range(60):
    opt.zero_grad()
    contact     = body[foot, 1].abs().mean()                 # feet ON the floor (y=0)
    penetration = (-scene_sdf(body)).clamp(min=0).mean()      # not INSIDE the obstacle
    (contact + penetration).backward(); opt.step()           # physics = free 3D supervision

print("mean foot height (->0):", round(body[foot, 1].abs().mean().item(), 3))
print("vertices still penetrating:", int((scene_sdf(body) < 0).sum()))

# Affordance: which locations are 'sit-able' = free space at sitting height.
grid = torch.rand(200, 3); grid[:, 1] = 0.5
print("sit-able locations (free space):", int((scene_sdf(grid) > 0.1).sum()))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A8
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks